In [1]:
!unzip -q -o ./data/you-tube-comments-signal-vsosai.zip -d ./data/

In [2]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

## Dependencies

In [2]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier, Pool

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
import seaborn

import emoji

from tqdm.cli import tqdm
from tqdm import trange

tqdm.pandas()

import string

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

from gensim.models import Word2Vec

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('names', quiet=True)


True

In [3]:
df_train = pd.read_csv('./data/train.csv')
df_test = pd.read_csv('./data/test.csv')
df_train.head()

,id,comment_text,target
0,59,Eh c kii ce favoritisme lui il a tou led iphon...,1
1,167,It's a myth that Neanderthals were stupid. The...,1
2,169,Emma is super white.\nIn other news...,1
3,372,Chelsea and Aubrey are my favourite on Teen Mo...,1
4,373,that mustard colour tho 😻😻 p.s absolutely love...,1


## EDA

In [4]:
df_train.shape

(51153, 3)

In [5]:
df_train['target'].value_counts()

target
0    38530
1     7808
2     3125
3     1690
Name: count, dtype: int64

In [6]:
df_train['comment_text'][:10]

0    Eh c kii ce favoritisme lui il a tou led iphon...
1    It's a myth that Neanderthals were stupid. The...
2               Emma is super white.\nIn other news...
3    Chelsea and Aubrey are my favourite on Teen Mo...
4    that mustard colour tho 😻😻 p.s absolutely love...
5    OMG u tagged Dina, never see anyone mention he...
6    when you said you're girl twins would be calle...
7                                 Loved this lousie♥♥♥
8    I'm a man united fan but anyone who can't see ...
9    When Christensen speaks better English than Ro...
Name: comment_text, dtype: object

In [7]:
df_train.isna().sum()

id              0
comment_text    9
target          0
dtype: int64

In [8]:
df_train = df_train.fillna("NA")
df_test = df_test.fillna("NA")

## Solutions

In [9]:
# def text_feature_extraction(df: pd.DataFrame, col: str):
#     df['emoji_count'] = df[col].progress_apply(lambda x: len(emoji.emoji_list(x)))
#     df['happy'] = df[col].progress_apply(lambda x: x.count(':)') + x.count(';)') + x.count('(:') + x.count('(;'))
#     df['sad'] = df[col].progress_apply(lambda x: x.count(':(') + x.count(';(') + x.count('):') + x.count(');'))
#     df['is_na'] = df[col] == "NA"
#     return df

In [10]:
# def text_normalize(text: str):
#     text = text.lower()
#     text = text.replace('\n', ' ')
#     text = text.replace('\r', ' ')
#     text = text.replace('\t', ' ')
#     text = "".join([ch for ch in text if ch not in string.punctuation])
#     text = emoji.replace_emoji(text, replace='')

#     words = word_tokenize(text)
#     stop_words = set(stopwords.words('english'))
#     words = [word for word in words if word not in stop_words]
    
#     stemmer = PorterStemmer()
#     words = [stemmer.stem(word) for word in words]
#     text = ' '.join(words)
#     return text
    
# text_normalize(df_train['comment_text'][4])

In [ ]:
import re
import numpy as np

def text_feature_extraction(df: pd.DataFrame, col: str):
    df['emoji_count'] = df[col].progress_apply(lambda x: len(emoji.emoji_list(x)))
    df['happy'] = df[col].progress_apply(lambda x: x.count(':)') + x.count(';)') + x.count('(:') + x.count('(;'))
    df['sad'] = df[col].progress_apply(lambda x: x.count(':(') + x.count(';(') + x.count('):') + x.count(');'))
    df['is_na'] = df[col] == "NA"

    URL_RE      = r'https?://\S+|www\.\S+'
    EMAIL_RE    = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    NUMBER_RE   = r'-?\d+\.?\d*'
    MENTION_RE  = r'@\w+'
    HASHTAG_RE  = r'#\w+'
    PHONE_RE    = r'[\+]?[(]?[0-9]{1,4}[)]?[-\s\./0-9]{7,}'
    LAUGH_RE    = r'\b(?:h[ae]h[ae]h?[ae]?h?|lo+l+|lma+o+|rof+l+|хa*ха+х*а*)\b'
    REPEAT_PUNCT_RE = r'([!?.]){2,}'                
    ELONGATED_RE    = r'\b\w*(.)\1{2,}\w*\b'         
    CAPS_WORD_RE    = r'\b[A-Z]{2,}\b'                

    df['url_count']       = df[col].progress_apply(lambda x: len(re.findall(URL_RE, x)))
    df['email_count']     = df[col].progress_apply(lambda x: len(re.findall(EMAIL_RE, x)))
    df['number_count']    = df[col].progress_apply(lambda x: len(re.findall(NUMBER_RE, x)))
    df['mention_count']   = df[col].progress_apply(lambda x: len(re.findall(MENTION_RE, x)))
    df['hashtag_count']   = df[col].progress_apply(lambda x: len(re.findall(HASHTAG_RE, x)))
    df['phone_count']     = df[col].progress_apply(lambda x: len(re.findall(PHONE_RE, x)))
    df['laugh_count']     = df[col].progress_apply(lambda x: len(re.findall(LAUGH_RE, x, re.I)))
    df['repeat_punct']    = df[col].progress_apply(lambda x: len(re.findall(REPEAT_PUNCT_RE, x)))
    df['elongated_count'] = df[col].progress_apply(lambda x: len(re.findall(ELONGATED_RE, x, re.I)))
    df['caps_count']      = df[col].progress_apply(lambda x: len(re.findall(CAPS_WORD_RE, x)))

    def _number_stats(text):
        nums = [float(n) for n in re.findall(NUMBER_RE, text)]
        if not nums:
            return 0.0, 0.0, 0.0
        return float(np.mean(nums)), float(np.sum(nums)), float(np.max(nums))

    stats = df[col].progress_apply(lambda x: _number_stats(x))
    df['number_mean'] = stats.apply(lambda x: x[0])
    df['number_sum']  = stats.apply(lambda x: x[1])
    df['number_max']  = stats.apply(lambda x: x[2])

    df['char_count']      = df[col].progress_apply(len)
    df['word_count']      = df[col].progress_apply(lambda x: len(x.split()))
    df['avg_word_len']    = df[col].progress_apply(
        lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
    )
    df['unique_word_ratio'] = df[col].progress_apply(
        lambda x: len(set(x.lower().split())) / max(len(x.split()), 1)
    )
    df['upper_char_ratio'] = df[col].progress_apply(
        lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1)
    )
    df['exclamation_count'] = df[col].progress_apply(lambda x: x.count('!'))
    df['question_count']    = df[col].progress_apply(lambda x: x.count('?'))
    df['newline_count']     = df[col].progress_apply(lambda x: x.count('\n'))

    return df


def text_normalize(text: str) -> str:
    text = re.sub(r'https?://\S+|www\.\S+', ' __URL__ ', text)

    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', ' __EMAIL__ ', text)

    text = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', ' __IP__ ', text)

    text = re.sub(r'[\+]?[(]?[0-9]{1,4}[)]?[-\s\./0-9]{7,}', ' __PHONE__ ', text)

    text = re.sub(r'\b\d{1,2}[/\-\.]\d{1,2}[/\-\.]\d{2,4}\b', ' __DATE__ ', text)

    text = re.sub(r'\b\d{1,2}:\d{2}(?::\d{2})?\s*(?:am|pm)?\b', ' __TIME__ ', text, flags=re.I)

    text = re.sub(r'-?\b\d+\.?\d*\b', ' __NUMBER__ ', text)

    text = re.sub(r'@\w+', ' __USER__ ', text)

    text = re.sub(r'#(\w+)', r' __HASHTAG__ \1 ', text)

    text = re.sub(
        r'\b(?:h[ae](?:h[ae])+h?|a?ha+h+a*|lo+l+z*|lma+o+|rof+l+|хa*(?:ха)+х*а*)\b',
        ' __LAUGH__ ', text, flags=re.I
    )

    text = re.sub(r'([!?.]){2,}', r'\1 __REPEAT__ ', text)

    text = re.sub(r'(.)\1{2,}', r'\1\1 __ELONG__ ', text)

    def _caps_tag(m):
        return ' __CAPS__ ' + m.group(0).lower() + ' '
    text = re.sub(r'\b[A-Z]{2,}\b', _caps_tag, text)

    text = emoji.replace_emoji(text, replace=' __EMOJI__ ')

    text = text.lower()
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    text = re.sub(r'[^\w\s]', '', text)

    words = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    words = [w for w in words if w not in stop_words or w.startswith('__')]

    stemmer = PorterStemmer()
    words = [w if w.startswith('__') and w.endswith('__') else stemmer.stem(w) for w in words]

    text = ' '.join(words)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [12]:
word_tokenize("<sun>")

['<', 'sun', '>']

In [13]:
df_extra = pd.read_csv('./data/extra_unlabeled.csv').dropna()

In [14]:
df_train = text_feature_extraction(df_train, 'comment_text')
df_test = text_feature_extraction(df_test, 'comment_text')

100%|██████████| 15050/15050 [00:00<00:00, 2052955.48it/s]


In [15]:
df_train_clean = df_train.copy()
df_test_clean = df_test.copy()
df_extra_clean = df_extra.copy()

# df_train_clean = text_feature_extraction(df_train_clean, 'comment_text')
df_train_clean['comment_text'] = df_train_clean['comment_text'].progress_apply(text_normalize)

# df_test_clean = text_feature_extraction(df_test_clean, 'comment_text')
df_test_clean['comment_text'] = df_test_clean['comment_text'].progress_apply(text_normalize)

# df_extra_clean = text_feature_extraction(df_extra_clean, 'comment_text')
df_extra_clean['comment_text'] = df_extra_clean['comment_text'].progress_apply(text_normalize)

df_train_clean.head()

  1%|          | 267/51153 [00:00<00:19, 2669.26it/s]

100%|██████████| 652233/652233 [02:36<00:00, 4165.09it/s]


,id,comment_text,target,emoji_count,happy,sad,is_na,url_count,email_count,number_count,...,number_sum,number_max,char_count,word_count,avg_word_len,unique_word_ratio,upper_char_ratio,exclamation_count,question_count,newline_count
0,59,eh c kii ce favoritism lui il tou led iphon ap...,1,0,0,0,False,0,0,0,...,0.0,0.0,185,37,4.027027,0.945946,0.005405,0,0,0
1,167,myth neanderth stupid actual brainpow human ti...,1,0,0,0,False,0,0,0,...,0.0,0.0,129,22,4.909091,0.909091,0.023256,0,0,0
2,169,emma super whitenin news < < cap > repeat >,1,0,0,0,False,0,0,0,...,0.0,0.0,38,6,5.500000,1.000000,0.052632,0,0,0
3,372,chelsea aubrey favourit teen mom well < emoji ...,1,4,0,0,False,0,0,0,...,0.0,0.0,100,20,4.050000,0.950000,0.050000,1,0,0
4,373,mustard colour tho < emoji > < emoji > ps abso...,1,2,1,0,False,0,0,0,...,0.0,0.0,149,24,5.250000,1.000000,0.006711,0,0,0


### CatBoost

In [16]:
X = df_train_clean.drop(columns=['id', 'target'])
y = df_train_clean['target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [17]:
train_pool = Pool(data=X_train, label=y_train, text_features=['comment_text'])
val_pool = Pool(data=X_val, label=y_val, text_features=['comment_text'])

In [18]:
model = CatBoostClassifier(
    iterations=50000,
    learning_rate=0.01,
    eval_metric='MultiClass',
    random_seed=42,
    early_stopping_rounds=1000,
    use_best_model=True,
    thread_count=-1,
    task_type='GPU',
    auto_class_weights='Balanced',
    text_features=['comment_text']
)

model.fit(
   train_pool,
   eval_set=val_pool,
   verbose=1000
)

0:	learn: 1.3742352	test: 1.3734215	best: 1.3734215 (0)	total: 8.39ms	remaining: 6m 59s
1000:	learn: 0.5912959	test: 0.5515145	best: 0.5515145 (1000)	total: 3.94s	remaining: 3m 12s
2000:	learn: 0.5259556	test: 0.5045503	best: 0.5045503 (2000)	total: 7.53s	remaining: 3m
3000:	learn: 0.4844555	test: 0.4790511	best: 0.4790511 (3000)	total: 11.1s	remaining: 2m 54s
4000:	learn: 0.4511248	test: 0.4600205	best: 0.4600205 (4000)	total: 14.7s	remaining: 2m 49s
5000:	learn: 0.4239922	test: 0.4454315	best: 0.4454315 (5000)	total: 18.3s	remaining: 2m 44s
6000:	learn: 0.4008642	test: 0.4340412	best: 0.4340412 (6000)	total: 21.9s	remaining: 2m 40s
7000:	learn: 0.3805594	test: 0.4241293	best: 0.4241293 (7000)	total: 25.5s	remaining: 2m 36s
8000:	learn: 0.3627362	test: 0.4152417	best: 0.4152417 (8000)	total: 29.1s	remaining: 2m 32s
9000:	learn: 0.3467671	test: 0.4073213	best: 0.4073213 (9000)	total: 32.8s	remaining: 2m 29s
10000:	learn: 0.3323552	test: 0.4009132	best: 0.4009132 (10000)	total: 36.4s	re

CatBoostClassifier(auto_class_weights='Balanced', early_stopping_rounds=1000, eval_metric='MultiClass', iterations=50000, learning_rate=0.01, random_seed=42, task_type='GPU', text_features=['comment_text'], use_best_model=True)

In [19]:
subm = df_test_clean.copy()
subm['target'] = model.predict(df_test_clean.drop(columns=['id']))
subm[['id', 'target']].to_csv("subm.csv", index=False)
subm.head()

,id,comment_text,emoji_count,happy,sad,is_na,url_count,email_count,number_count,mention_count,...,number_max,char_count,word_count,avg_word_len,unique_word_ratio,upper_char_ratio,exclamation_count,question_count,newline_count,target
0,200,arab never convers rel dad side alway turn awk...,0,0,0,False,0,0,0,0,...,0.0,199,38,4.263158,0.894737,0.025126,0,0,0,2
1,607,could subscrib dude perfect million time would,0,0,0,False,0,0,0,0,...,0.0,60,12,4.083333,0.916667,0.066667,0,0,0,1
2,1100,fortun say live era harri fuckin style forev a...,1,0,0,False,0,0,0,0,...,0.0,118,25,3.760000,0.880000,0.025424,0,0,0,1
3,1137,easi classic cover band great talent,0,0,0,False,0,0,0,0,...,0.0,66,13,4.153846,1.000000,0.030303,0,0,0,1
4,1139,realli realli realli love cover hope harri sin...,0,0,0,False,0,0,0,0,...,0.0,154,30,4.166667,0.800000,0.058442,0,0,0,1


### LogReg

In [20]:
df_train.head()

,id,comment_text,target,emoji_count,happy,sad,is_na,url_count,email_count,number_count,...,number_sum,number_max,char_count,word_count,avg_word_len,unique_word_ratio,upper_char_ratio,exclamation_count,question_count,newline_count
0,59,Eh c kii ce favoritisme lui il a tou led iphon...,1,0,0,0,False,0,0,0,...,0.0,0.0,185,37,4.027027,0.945946,0.005405,0,0,0
1,167,It's a myth that Neanderthals were stupid. The...,1,0,0,0,False,0,0,0,...,0.0,0.0,129,22,4.909091,0.909091,0.023256,0,0,0
2,169,Emma is super white.\nIn other news...,1,0,0,0,False,0,0,0,...,0.0,0.0,38,6,5.500000,1.000000,0.052632,0,0,0
3,372,Chelsea and Aubrey are my favourite on Teen Mo...,1,4,0,0,False,0,0,0,...,0.0,0.0,100,20,4.050000,0.950000,0.050000,1,0,0
4,373,that mustard colour tho 😻😻 p.s absolutely love...,1,2,1,0,False,0,0,0,...,0.0,0.0,149,24,5.250000,1.000000,0.006711,0,0,0


In [ ]:
all_text = df_train_clean['comment_text'].tolist() + df_extra_clean['comment_text'].tolist() + df_test_clean['comment_text'].tolist()
all_text[:10]

['eh c kii ce favoritism lui il tou led iphon appl watch ipad macbook en avanc est gratuit wallah je reflechi une ide poir wow ador je sui rich mdrr sa arrivera qau autr',
 'myth neanderth stupid actual brainpow human time got assimil',
 'emma super whitenin news < < cap > repeat >',
 'chelsea aubrey favourit teen mom well < emoji > < emoji > especi cole babi < emoji > < emoji >',
 'mustard colour tho < emoji > < emoji > ps absolut love video whenev someon mention babi name instantli think sim < < cap > repeat > okayi < < cap > elong >',
 '< cap > omg u tag dina never see anyon mention even though she amaz',
 'said your girl twin would call lyla luci sister liter look like omg name lyla < emoji >',
 'love lousi < emoji > < emoji > < < cap > elong >',
 'im man unit fan anyon cant see hazard best player prem last season mad',
 'christensen speak better english rooney']

In [22]:
# tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 3), stop_words='english')
# tfidf.fit(all_text)

In [23]:
X_clean = df_train_clean['comment_text']
X_clean_test = df_test_clean['comment_text']
y = df_train_clean['target']

# X_tfidf_train, X_tfidf_val, y_train, y_val = train_test_split(X_tfidf, y, test_size=0.3, random_state=42, stratify=y)

# svd = TruncatedSVD(n_components=2000, random_state=42)
# X_tfidf_train = svd.fit_transform(X_tfidf_train)
# X_tfidf_val = svd.transform(X_tfidf_val)
# X_tfidf_test = svd.transform(X_tfidf_test)

# sc = StandardScaler()
# X_tfidf_train = sc.fit_transform(X_tfidf_train)
# X_tfidf_val = sc.transform(X_tfidf_val)
# X_tfidf_test = sc.transform(X_tfidf_test)

# print(f"TF-IDF матрица: {X_clean.shape}")
# print(f"Ненулевых элементов: {X_clean.nonzero()[0].shape[0] / np.prod(X_clean.shape) * 100:.2f}%")

In [24]:
# model = LogisticRegression(
#     max_iter=10000,
#     n_jobs=-1,
#     class_weight='balanced',
#     random_state=42
# )
# model.fit(X_tfidf_train, y_train)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 3), stop_words='english')),
    ('clf', LogisticRegression(max_iter=10000, class_weight='balanced', random_state=42))
])

# y_pred = model.predict(X_tfidf_val)
# f1_score(y_val, y_pred, average='macro')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X_clean, y, cv=cv, scoring='f1_macro', n_jobs=-1)
scores.mean()

np.float64(0.6720487834256896)

In [25]:
# subm = df_test.copy()
# subm['target'] = model.predict(X_tfidf_test)
# subm.head()

# subm[['id', 'target']].to_csv("subm.csv", index=False)

In [26]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 3), stop_words='english')
tfidf.fit(all_text)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [27]:
X_tfidf = tfidf.transform(X_clean)
X_tfidf_test = tfidf.transform(X_clean_test)

In [28]:
model = LogisticRegression(
    max_iter=1000000,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
model.fit(X_tfidf, y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000000
,multi_class,'deprecated'


In [29]:
subm = df_test.copy()
subm['target'] = model.predict(X_tfidf_test)
subm.head()

subm[['id', 'target']].to_csv("subm.csv", index=False)

### Catboost with TF-idf

### Word2Vec

In [30]:
# w2v_model = Word2Vec(
#     all_text_words,
#     vector_size=32,
#     min_count=5,
#     window=5,
#     workers=-1
# ).wv


In [31]:
# w2v_model.most_similar("bad")

In [32]:
def text_to_vector(text, model):
    words = word_tokenize(text.lower())
    words = [w for w in words if w in model.key_to_index]
    if not words:
        return np.zeros(model.vector_size)
    vectors = [model[w] for w in words]
    return np.mean(vectors, axis=0)

In [33]:
# X_w2v = np.vstack(X['comment_text'].progress_apply(lambda x: text_to_vector(x, w2v_model)))

In [34]:
# X_w2v[0]

In [35]:
# X_w2v_train, X_w2v_val, y_train, y_val = train_test_split(X_w2v, y, test_size=0.3, shuffle=True, random_state=42, stratify=y)

In [36]:
# model = CatBoostClassifier(
#     iterations=50000,
#     learning_rate=0.01,
#     eval_metric='MultiClass',
#     random_seed=42,
#     early_stopping_rounds=1000,
#     use_best_model=True,
#     thread_count=-1,
#     task_type='GPU',
#     auto_class_weights='Balanced',
# )

# model.fit(
#    X_w2v_train,
#    y_train,
#    eval_set=(X_w2v_val, y_val),
#    verbose=1000
# )

In [37]:
# model = LogisticRegression(
#     max_iter=1000000,
#     class_weight='balanced',
#     n_jobs=-1,
#     random_state=42
# )

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# scores = cross_val_score(model, X_w2v, y, cv=cv, scoring='f1_macro', n_jobs=-1)
# scores.mean()